# TRELLIS.2 GGUF — Colab T4

**Среда выполнения → Сменить среду выполнения → T4 GPU. Две основные ячейки: «Установка» и «Запуск / перезапуск».**

Установку выполните один раз на новую среду. Она сама последовательно пройдёт все этапы без промежуточных кликов: окружение, CUDA-сборки и проверки. Первичная сборка может занять десятки минут. Для каждого запуска или перезапуска сервера используйте только вторую ячейку — она не устанавливает пакеты и не повторяет сборки. Ключи GitHub/Hugging Face не нужны для предусмотренных публичных источников.

Основано на вашем Trellis2_GGUF_Colab(11).ipynb. Исправления: фиксированные Python/Torch/CUDA, обязательный GGUF-загрузчик, отказ от несовместимых готовых wheels, CUDA smoke tests, отменяемое удаление фона, HTTP/2 и запуск без занятой ячейки. Исходные Git-ревизии закреплены в colab/sources.lock.json. Зависимости Python разрешаются с жёсткими ограничениями Torch; полный список сохраняется в /content/t4-environment.txt.

В интерфейсе откройте **Workflows → Trellis2-T4 → Only_Mesh_Simple** и загрузите свою картинку. Для PNG с прозрачностью повторное удаление фона пропускается. Модели Trellis/DINO скачиваются при первом запуске workflow; это не время загрузки самого интерфейса.

Примечание: реальные генерация/GLB-экспорт на T4 ещё должны быть проверены. CUDA-проверки остановят запуск при известной несовместимости. Qwen-вариант требует дополнительных сторонних нод/весов и не является базовым профилем T4. Дополнительный nvdiffrec PBR-preview в эту установку не включён.


## 1. Установка — один раз на новую среду

In [ ]:
# УСТАНОВКА. Выберите T4 в настройках среды и нажмите ▶ здесь.
# Первый запуск собирает CUDA-модули. Повторный использует сохранённые этапы.
PRELOAD_BACKGROUND_MODEL = True

import base64, hashlib, io, json, os, pathlib, subprocess, sys, time, urllib.request, zipfile
ROOT = pathlib.Path('/content')
if not ROOT.exists():
    raise RuntimeError('Этот notebook предназначен для Google Colab с T4 GPU.')
APP = ROOT / 'trellis-t4-bootstrap'
APP.mkdir(exist_ok=True)
PAYLOAD = '...'
bundle = base64.b64decode(PAYLOAD)
fingerprint = hashlib.sha256(bundle).hexdigest()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for item in archive.infolist():
        if not (APP/item.filename).resolve().is_relative_to(APP.resolve()):
            raise ValueError('Некорректный путь в пакете')
    archive.extractall(APP)

def command(args, timeout=None):
    # Colab needs a PIPE + print: inherited file descriptors can hide subprocess logs.
    sys.path.insert(0,str(APP/'colab'))
    from process_output import run_streamed
    name=str(args[-1])
    return run_streamed(args,ROOT/f'colab-install-{name}.log',timeout=timeout)

def main():
    state = ROOT/'trellis-t4-processes.json'
    if state.exists():
        raise RuntimeError('Сначала остановите ComfyUI ячейкой остановки. Для обычного перезапуска используйте ячейку запуска, а не установку.')
    marker_dir=ROOT/'t4-stage-cache';marker_dir.mkdir(exist_ok=True)
    stages=[('system','1/5 — Python 3.12, Torch cu126, CUDA-компилятор 12.6'),
            ('dependencies','2/5 — Репозитории, ComfyUI-GGUF, зависимости и профили'),
            ('builds','3/5 — Сборка CUDA-расширений для T4 (одна за другой)')]
    for name,title in stages:
        marker=marker_dir/(name+'.json')
        complete=False
        if marker.exists():
            try:complete=json.loads(marker.read_text()).get('fingerprint')==fingerprint
            except ValueError:pass
        print('\n'+title,flush=True)
        if complete and pathlib.Path('/content/venv312/bin/python').exists():
            print('Этап уже завершён, использую кэш.',flush=True)
        else:
            # Invalidates the verification flag before changing any packages.
            (ROOT/'t4-verified.json').unlink(missing_ok=True)
            command([sys.executable,APP/'colab/setup_t4.py',name])
            marker.write_text(json.dumps({'fingerprint':fingerprint}))
    print('\n4/5 — Настоящие CUDA-операции, проверка ядра flex_gemm',flush=True)
    command([sys.executable,APP/'colab/setup_t4.py','verify'])
    print('\n5/5 — Подготовка удаления фона',flush=True)
    if PRELOAD_BACKGROUND_MODEL:
        env=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
        from process_output import run_streamed
        run_streamed(['/content/venv312/bin/python','-u','-c',
            "from rembg import new_session; print('Загрузка/проверка u2net...',flush=True); new_session('u2net',providers=['CPUExecutionProvider']); print('Удаление фона готово.',flush=True)"],
            ROOT/'colab-install-background.log',env=env,timeout=180)
    print('\nУстановка готова. Теперь выполните отдельную ячейку «Запуск / перезапуск ComfyUI».')

try:
    main()
except KeyboardInterrupt:
    # Stop only the processes owned by this notebook, if they were started.
    subprocess.run([sys.executable,str(APP/'colab/server_t4.py'),'stop'])
    print('Остановлено. Завершённые этапы сохранены; нажмите ▶ для продолжения.')
    raise
except Exception:
    print('Этап завершился ошибкой. Причина указана выше. После исправления нажмите ▶ снова; завершённые этапы будут пропущены.')
    raise


## 2. Запуск / перезапуск ComfyUI
Запускает только сервер и новый HTTP/2-туннель. Если сервер этого notebook уже работает, он будет перезапущен. Ячейка завершится после появления ссылки; ComfyUI продолжит работать и не займёт очередь выполнения ячеек. При удалении всей среды Colab сначала снова потребуется установка.


In [ ]:
from pathlib import Path
import sys
launcher=Path('/content/trellis-t4-bootstrap/colab/server_t4.py')
if not launcher.exists():
    raise RuntimeError('Сначала выполните установку в этой среде Colab.')
sys.path.insert(0,str(launcher.parent))
from process_output import run_streamed
run_streamed([sys.executable,'-u',str(launcher),'start'],'/content/colab-launch.log')


## Необязательно: остановить только ComfyUI и туннель
Отметьте STOP_COMFY и выполните эту ячейку. Установленные пакеты и модели сохранятся до удаления среды Colab. После остановки используйте ячейку 2 для повторного запуска. По умолчанию эта ячейка ничего не останавливает при «Выполнить все».


In [ ]:
STOP_COMFY = False # @param {type:"boolean"}
if STOP_COMFY:
    import subprocess, sys
    subprocess.run([sys.executable, '/content/trellis-t4-bootstrap/colab/server_t4.py', 'stop'], check=True)


## Необязательно: последние строки логов

In [ ]:
from pathlib import Path
for name in ['comfy-t4.log','cloudflare-t4.log']:
    p=Path('/content')/name
    print('\n'+name+'\n'+(p.read_text(errors='replace')[-12000:] if p.exists() else 'Лог ещё не создан'))


## Необязательно: скачать результаты
Перед завершением Colab сохраните GLB на компьютер. /content не является постоянным хранилищем. ZIP создаётся вне папки output.


In [ ]:
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    from pathlib import Path
    import shutil
    from google.colab import files
    out=Path('/content/ComfyUI/output')
    if not out.exists():
        raise RuntimeError('Папка результатов ещё не создана')
    archive=shutil.make_archive('/content/trellis-results','zip',out)
    files.download(archive)
